In [111]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import lightgbm as lgb
import random
from sklearn.metrics import f1_score
from matplotlib import pyplot as plt
from sklearn.cluster import KMeans
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, coo_matrix
from scipy.sparse.linalg import svds
from scipy.spatial.distance import cosine
from sklearn.decomposition import NMF
import networkx as nx

In [2]:
version = "v6"

In [3]:
orders = pd.read_csv("orders.csv")
vendors = pd.read_csv("vendors.csv")
train_customers = pd.read_csv("train_customers.csv")
train_locations = pd.read_csv("train_locations.csv")
test_customers = pd.read_csv("test_customers.csv")
test_locations = pd.read_csv("test_locations.csv")
sample_submission = pd.read_csv("SampleSubmission.csv")
validation_k_folds = pd.read_csv("validation_7_folds.csv")

/home/ubuntu/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (15,16,18,19,20) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [4]:
vendors['primary_tags'] = vendors['primary_tags'].fillna('{"primary_tags":"0"}').map(lambda x : 
                                                                                     int(json.loads(x)['primary_tags']))
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(vendors.vendor_tag.astype(str).map(lambda x : " ".join(x.split(","))))
U, sigma, Vt = svds(X, k = 8)
vendor_tags = pd.DataFrame(U)
vendor_tags.columns = ["vendor_tags_svd_" + str(c) for c in vendor_tags.columns]
vendors = pd.concat([vendors, vendor_tags], axis = 1)

In [5]:
max_created_date = pd.to_datetime(vendors.created_at).max()
member_since = (max_created_date - pd.to_datetime(vendors[['id', 'created_at']].set_index('id').created_at)).dt.days.to_dict()
vendors['vendor_member_since'] = vendors['id'].map(member_since)

In [15]:
vendors['vendor_updated_days_after_created'] =  (pd.to_datetime(vendors['updated_at']) - pd.to_datetime(vendors['created_at'])).dt.days

In [18]:
max_updated_date = pd.to_datetime(vendors.updated_at).max()
updated_since = (max_updated_date - pd.to_datetime(vendors[['id', 'updated_at']].set_index('id').updated_at)).dt.days.to_dict()
vendors['vendor_updated_since'] = vendors['id'].map(updated_since)

In [20]:
for var in tqdm(['sunday_from_time1', 'sunday_to_time1',
       'sunday_from_time2', 'sunday_to_time2', 'monday_from_time1',
       'monday_to_time1', 'monday_from_time2', 'monday_to_time2',
       'tuesday_from_time1', 'tuesday_to_time1', 'tuesday_from_time2',
       'tuesday_to_time2', 'wednesday_from_time1', 'wednesday_to_time1',
       'wednesday_from_time2', 'wednesday_to_time2', 'thursday_from_time1',
       'thursday_to_time1', 'thursday_from_time2', 'thursday_to_time2',
       'friday_from_time1', 'friday_to_time1', 'friday_from_time2',
       'friday_to_time2', 'saturday_from_time1', 'saturday_to_time1',
       'saturday_from_time2', 'saturday_to_time2']):

    vendors[var] = pd.to_datetime(vendors[var], format = "%H:%M:%S").map(lambda x : x.hour + x.minute/60)

In [21]:
train_locations['key'] = 1
test_locations['key'] = 1
vendors['key'] = 1

target = orders[['customer_id', 'LOCATION_NUMBER', 'vendor_id']].drop_duplicates().rename(
                                                                columns = {'LOCATION_NUMBER' : 'location_number'})
target['target'] = 1
train = train_locations.merge(vendors.rename(columns = {'id' : 'vendor_id', 'latitude' : 'vendor_latitude', 
                        'longitude' : 'vendor_longitude'}), on = 'key', how = 'inner').merge(
                target, on = ['customer_id', 'location_number', 'vendor_id'], how = 'left').drop('key', axis = 1)
test = test_locations.merge(vendors.rename(columns = {'id' : 'vendor_id', 'latitude' : 'vendor_latitude', 
                                'longitude' : 'vendor_longitude'}), on = 'key', how = 'inner').drop('key', axis = 1)

train.target.fillna(0, inplace = True)
train.shape, test.shape

((5950300, 76), (1672000, 75))

In [33]:
customer_is_female = train_customers[['akeed_customer_id', 'gender']].append(test_customers[['akeed_customer_id', 'gender']]
                ).set_index('akeed_customer_id').gender.astype(str).map(lambda x : 1 if "female" in x.lower() else 0).to_dict()
train['customer_is_female'] = train['customer_id'].map(customer_is_female)
test['customer_is_female'] = test['customer_id'].map(customer_is_female)


In [34]:
customer_age = train_customers[['akeed_customer_id', 'dob']].append(test_customers[['akeed_customer_id', 'dob']]
                ).set_index('akeed_customer_id').dob.map(lambda x : 2020 - x).to_dict()
train['customer_age'] = train['customer_id'].map(customer_age)
test['customer_age'] = test['customer_id'].map(customer_age)


In [35]:
max_created_date = pd.to_datetime(train_customers.created_at.append(test_customers.created_at)).max()
member_since = (max_created_date - pd.to_datetime(train_customers[['akeed_customer_id', 'created_at']].append(test_customers[['akeed_customer_id', 'created_at']]
                ).set_index('akeed_customer_id').created_at)).dt.days.to_dict()
train['member_since'] = train['customer_id'].map(member_since)
test['member_since'] = test['customer_id'].map(member_since)

In [38]:
max_updated_date = pd.to_datetime(train_customers.updated_at.append(test_customers.updated_at)).max()
updated_since = (max_updated_date - pd.to_datetime(train_customers[['akeed_customer_id', 'updated_at']].append(test_customers[['akeed_customer_id', 'updated_at']]
                ).set_index('akeed_customer_id').updated_at)).dt.days.to_dict()
train['updated_since'] = train['customer_id'].map(updated_since)
test['updated_since'] = test['customer_id'].map(updated_since)

In [45]:
df = train_customers.append(test_customers)
df['last_update_date'] = df[['akeed_customer_id', 'updated_at']].groupby('akeed_customer_id').transform('max')
df['updated_created_diff'] = (pd.to_datetime(df['last_update_date']) - pd.to_datetime(df.created_at)).dt.days
updated_created_diff = df.set_index('akeed_customer_id').updated_created_diff.to_dict()
train['updated_created_diff'] = train.customer_id.map(updated_created_diff)
test['updated_created_diff'] = test.customer_id.map(updated_created_diff)


In [51]:
customer_num_updates = train_customers.akeed_customer_id.append(test_customers.akeed_customer_id).value_counts().to_dict()
train['customer_num_updates'] = train.customer_id.map(customer_num_updates)
test['customer_num_updates'] = test.customer_id.map(customer_num_updates)

In [52]:
def correct_lat(x):
    if x < -1000:
        return x  + 1038
    if x < -660:
        return x + 668
    if x < -500:
        return x + 535
    if x  < -150:
        return x + 154
    if x < -90:
        return x + 96
    if x < -50:
        return x + 49
    if x < -30:
        return x + 31
    if x < -55:
        return x + 60
    if x < -55:
        return x + 60
    if x > 8 and x < 16:
        return x - 12.5
    if x > 16 and x < 30:
        return x - 19
    if x > 90 and x < 100:
        return x - 97
    if x > 120 and x < 130:
        return x - 125.5
    if x > 150 and x < 160:
        return x - 155
    if x > 200 and x < 210:
        return x - 204
    if x > 280 and x < 300:
        return x - 284
    if x > 450 and x < 460:
        return x - 453
    return x
    
    
def correct_lon(x):
    if x < -130:
        return x  + 132
    if x < -100:
        return x + 102
    if x < -75:
        return x + 78
    if x  < -65:
        return x + 67
    if x < -55:
        return x + 60
    if x < -7:
        return x + 9
    if x < -2:
        return x + 3
    if x > 10 and x < 50:
        return x - 44
    return x


In [53]:
df = train_locations[train_locations.latitude.notnull()][['customer_id', 'location_number', 'latitude', 'longitude']]
kmeans = KMeans(n_clusters=16, random_state=0).fit(df[['latitude', 'longitude']].values)
df = train_locations[train_locations.latitude.notnull()][['customer_id', 'location_number', 'latitude', 'longitude']].append(
    test_locations[test_locations.latitude.notnull()][['customer_id', 'location_number', 'latitude', 'longitude']])
df['location_cluster'] = kmeans.predict(df[['latitude', 'longitude']].values)

In [54]:
location_clusters = df.set_index(['customer_id', 'location_number']).location_cluster.to_dict()
train['location_cluster'] = [location_clusters.get((row.customer_id, row.location_number), -1) for row in train.itertuples()]
test['location_cluster'] = [location_clusters.get((row.customer_id, row.location_number), -1) for row in test.itertuples()]

In [55]:
orders['location_cluster'] = [location_clusters.get((row.customer_id, row.LOCATION_NUMBER), -1) for row in orders.itertuples()]
vendor_location_cluster_order_count = orders[['vendor_id', 'location_cluster', 'akeed_order_id']].groupby(
                                        ['vendor_id', 'location_cluster']).count().akeed_order_id.to_dict()
train['vendor_location_cluster_order_count'] = [vendor_location_cluster_order_count.get((row.vendor_id, row.location_cluster), 0)
                                                for row in train.itertuples()]
test['vendor_location_cluster_order_count'] = [vendor_location_cluster_order_count.get((row.vendor_id, row.location_cluster), 0)
                                                for row in test.itertuples()]


In [56]:
train['longitude_corrected'] = train.longitude.map(correct_lon)
train['latitude_corrected'] = train.latitude.map(correct_lon)
test['longitude_corrected'] = test.longitude.map(correct_lon)
test['latitude_corrected'] = test.latitude.map(correct_lon)


train['vendor_longitude_corrected'] = train.vendor_longitude.map(correct_lon)
train['vendor_latitude_corrected'] = train.vendor_latitude.map(correct_lon)
test['vendor_longitude_corrected'] = test.vendor_longitude.map(correct_lon)
test['vendor_latitude_corrected'] = test.vendor_latitude.map(correct_lon)


In [57]:
def get_distance(lon1, lat1, lon2, lat2):
    return np.sqrt(((lat1 - lat2)**2 + (lon1 - lon2)**2)/2)

train['vendor_distance'] = [get_distance(row.longitude, row.latitude, row.vendor_longitude, row.vendor_latitude)
                           for row in tqdm(train.itertuples())]
test['vendor_distance'] = [get_distance(row.longitude, row.latitude, row.vendor_longitude, row.vendor_latitude)
                           for row in tqdm(test.itertuples())]

train['vendor_distance_corrected'] = [get_distance(row.longitude_corrected, row.latitude_corrected, row.vendor_longitude_corrected, row.vendor_latitude_corrected)
                           for row in tqdm(train.itertuples())]
test['vendor_distance_corrected'] = [get_distance(row.longitude_corrected, row.latitude_corrected, row.vendor_longitude_corrected, row.vendor_latitude_corrected)
                           for row in tqdm(test.itertuples())]

In [58]:
train['lon_diff'] = train['longitude'] - train['vendor_longitude']
test['lon_diff'] = test['longitude'] - test['vendor_longitude']


train['lat_diff'] = train['latitude'] - train['vendor_latitude']
test['lat_diff'] = test['latitude'] - test['vendor_latitude']


train['lon_diff_corrected'] = train['longitude_corrected'] - train['vendor_longitude_corrected']
test['lon_diff_corrected'] = test['longitude_corrected'] - test['vendor_longitude_corrected']


train['lat_diff_corrected'] = train['latitude_corrected'] - train['vendor_latitude_corrected']
test['lat_diff_corrected'] = test['latitude_corrected'] - test['vendor_latitude_corrected']


In [59]:
train['lon_diff_abs'] = abs(train['longitude'] - train['vendor_longitude'])
test['lon_diff_abs'] = abs(test['longitude'] - test['vendor_longitude'])

train['lat_diff_abs'] = abs(train['latitude'] - train['vendor_latitude'])
test['lat_diff_abs'] = abs(test['latitude'] - test['vendor_latitude'])


train['lon_diff_corrected_abs'] = abs(train['longitude_corrected'] - train['vendor_longitude_corrected'])
test['lon_diff_corrected_abs'] = abs(test['longitude_corrected'] - test['vendor_longitude_corrected'])


train['lat_diff_corrected_abs'] = abs(train['latitude_corrected'] - train['vendor_latitude_corrected'])
test['lat_diff_corrected_abs'] = abs(test['latitude_corrected'] - test['vendor_latitude_corrected'])


In [60]:
customer_location_counts = train_locations['customer_id'].append(test_locations['customer_id']).value_counts().to_dict()
train['customer_location_counts'] = train['customer_id'].map(customer_location_counts)
test['customer_location_counts'] = test['customer_id'].map(customer_location_counts)

In [61]:
nearby_radius = 0.25
num_vendors_nearby = train[train.vendor_distance < nearby_radius][['customer_id', 'location_number', 'vendor_id']].append(
                    test[test.vendor_distance < nearby_radius][['customer_id', 'location_number', 'vendor_id']]).groupby(
                    ['customer_id', 'location_number']).count().vendor_id.to_dict()
train['num_vendors_nearby'] = [num_vendors_nearby.get((row.customer_id, row.location_number), 0) for row in train.itertuples()]
test['num_vendors_nearby'] = [num_vendors_nearby.get((row.customer_id, row.location_number), 0) for row in test.itertuples()]


In [62]:
distance_rank = train[['customer_id', 'location_number', 'vendor_id', 'vendor_distance']].append(
                    test[['customer_id', 'location_number', 'vendor_id', 'vendor_distance']])
distance_rank['distance_rank'] = distance_rank[['customer_id', 'location_number', 'vendor_distance']
                                              ].groupby(['customer_id', 'location_number']).rank(method = 'min')
distance_rank = distance_rank.set_index(['customer_id', 'location_number', 'vendor_id']).distance_rank.to_dict()
train['distance_rank'] = [distance_rank.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in train.itertuples()]
test['distance_rank'] = [distance_rank.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in test.itertuples()]

In [63]:
distance_rank_lat = train[['customer_id', 'location_number', 'vendor_id', 'lat_diff_abs']].append(
                    test[['customer_id', 'location_number', 'vendor_id', 'lat_diff_abs']])
distance_rank_lat['distance_rank_lat'] = distance_rank_lat[['customer_id', 'location_number', 'lat_diff_abs']
                                              ].groupby(['customer_id', 'location_number']).rank(method = 'min')
distance_rank_lat = distance_rank_lat.set_index(['customer_id', 'location_number', 'vendor_id']).distance_rank_lat.to_dict()
train['distance_rank_lat'] = [distance_rank_lat.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in train.itertuples()]
test['distance_rank_lat'] = [distance_rank_lat.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in test.itertuples()]

In [64]:
distance_rank_lon = train[['customer_id', 'location_number', 'vendor_id', 'lon_diff_abs']].append(
                    test[['customer_id', 'location_number', 'vendor_id', 'lon_diff_abs']])
distance_rank_lon['distance_rank_lon'] = distance_rank_lon[['customer_id', 'location_number', 'lon_diff_abs']
                                              ].groupby(['customer_id', 'location_number']).rank(method = 'min')
distance_rank_lon = distance_rank_lon.set_index(['customer_id', 'location_number', 'vendor_id']).distance_rank_lon.to_dict()
train['distance_rank_lon'] = [distance_rank_lon.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in train.itertuples()]
test['distance_rank_lon'] = [distance_rank_lon.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in test.itertuples()]

In [65]:
vendor_uniuqe_customer_count = orders[['vendor_id', 'customer_id']].drop_duplicates().vendor_id.value_counts().to_dict()
train['vendor_uniuqe_customer_count'] = train['vendor_id'].map(vendor_uniuqe_customer_count)
test['vendor_uniuqe_customer_count'] = test['vendor_id'].map(vendor_uniuqe_customer_count)

In [66]:
vendor_uniuqe_customer_count = orders[['vendor_id', 'customer_id']].drop_duplicates().vendor_id.value_counts().to_dict()
train['vendor_uniuqe_customer_count'] = train['vendor_id'].map(vendor_uniuqe_customer_count)
test['vendor_uniuqe_customer_count'] = test['vendor_id'].map(vendor_uniuqe_customer_count)

In [67]:
df = orders[['vendor_id', 'customer_id', 'akeed_order_id']].groupby(['vendor_id', 'customer_id']).count().reset_index()
df['total_transaction'] = df[['vendor_id', 'akeed_order_id']].groupby('vendor_id').transform(sum)
df['customer_transaction_fraction'] = df['akeed_order_id'] / df['total_transaction']
df['plogp'] = -df.customer_transaction_fraction * df.customer_transaction_fraction.map(np.log)
vendor_entropy = df[['vendor_id', 'plogp']].groupby('vendor_id').sum().plogp.to_dict()

train['vendor_entropy'] = train.vendor_id.map(vendor_entropy)
test['vendor_entropy'] = test.vendor_id.map(vendor_entropy)


/home/ubuntu/anaconda3/lib/python3.7/site-packages/pandas/core/base.py:1154: RuntimeWarning: divide by zero encountered in log
  new_values = map_f(values, mapper)


In [68]:
vendor_total_amount_by_location_type = orders[['vendor_id', 'LOCATION_TYPE', 'grand_total']].fillna("nan").groupby(
                                    ['vendor_id', 'LOCATION_TYPE']).sum().grand_total.to_dict()
train['vendor_total_amount_by_location_type'] = [vendor_total_amount_by_location_type.get((row.vendor_id, str(row.location_type)), 0) for row in train.itertuples()]
test['vendor_total_amount_by_location_type'] = [vendor_total_amount_by_location_type.get((row.vendor_id, str(row.location_type)), 0) for row in test.itertuples()]

In [69]:
orders.delivery_date = pd.to_datetime(orders.delivery_date).dt.date
daily_order_counts = orders[['vendor_id', 'delivery_date', 'akeed_order_id']].groupby(['vendor_id', 'delivery_date']
                        ).count().reset_index()[['vendor_id', 'akeed_order_id']
                        ].groupby('vendor_id').agg(['mean', 'std']).akeed_order_id.to_dict()
train['daily_order_counts_mean'] = train.vendor_id.map(daily_order_counts['mean'])
train['daily_order_counts_std'] = train.vendor_id.map(daily_order_counts['std'])

test['daily_order_counts_mean'] = test.vendor_id.map(daily_order_counts['mean'])
test['daily_order_counts_std'] = test.vendor_id.map(daily_order_counts['std'])


In [70]:
vendor_ratings = orders[['vendor_id', 'vendor_rating']].groupby('vendor_id').agg(['mean', 'std', 'count']).vendor_rating.to_dict()
train['vendor_ratings_mean'] = train.vendor_id.map(vendor_ratings['mean'])
train['vendor_ratings_std'] = train.vendor_id.map(vendor_ratings['std'])
train['vendor_ratings_count'] = train.vendor_id.map(vendor_ratings['count'])

test['vendor_ratings_mean'] = test.vendor_id.map(vendor_ratings['mean'])
test['vendor_ratings_std'] = test.vendor_id.map(vendor_ratings['std'])
test['vendor_ratings_count'] = test.vendor_id.map(vendor_ratings['count'])


In [71]:
vendor_driver_ratings = orders[['vendor_id', 'driver_rating']].groupby('vendor_id').agg(['mean', 'std', 'count']).driver_rating.to_dict()
train['vendor_driver_ratings_mean'] = train.vendor_id.map(vendor_driver_ratings['mean'])
train['vendor_driver_ratings_std'] = train.vendor_id.map(vendor_driver_ratings['std'])
train['vendor_driver_ratings_count'] = train.vendor_id.map(vendor_driver_ratings['count'])

test['vendor_driver_ratings_mean'] = test.vendor_id.map(vendor_driver_ratings['mean'])
test['vendor_driver_ratings_std'] = test.vendor_id.map(vendor_driver_ratings['std'])
test['vendor_driver_ratings_count'] = test.vendor_id.map(vendor_driver_ratings['count'])


In [72]:
train['vendor_is_restaurant'] = train.vendor_category_en.map(lambda x : 1 if x == 'Restaurants' else 0)
test['vendor_is_restaurant'] = test.vendor_category_en.map(lambda x : 1 if x == 'Restaurants' else 0)

In [73]:
train.location_type.value_counts()
train['location_type_is_home'] = (train['location_type'] == "Home")*1
train['location_type_is_work'] = (train['location_type'] == "Work")*1
train['location_type_is_other'] = (train['location_type'] == "Other")*1

test['location_type_is_home'] = (test['location_type'] == "Home")*1
test['location_type_is_work'] = (test['location_type'] == "Work")*1
test['location_type_is_other'] = (test['location_type'] == "Other")*1


In [74]:
orders_delivery_distances = orders[['vendor_id', 'deliverydistance']].groupby('vendor_id').agg(['mean', 'std', 'max']).deliverydistance.to_dict()
train['vendor_mean_delivery_distance'] = train.vendor_id.map(orders_delivery_distances['mean'])
train['vendor_std_delivery_distance'] = train.vendor_id.map(orders_delivery_distances['std'])
train['vendor_max_delivery_distance'] = train.vendor_id.map(orders_delivery_distances['max'])


test['vendor_mean_delivery_distance'] = test.vendor_id.map(orders_delivery_distances['mean'])
test['vendor_std_delivery_distance'] = test.vendor_id.map(orders_delivery_distances['std'])
test['vendor_max_delivery_distance'] = test.vendor_id.map(orders_delivery_distances['max'])


In [75]:
cust_location_lat_range = (train_locations[['customer_id', 'latitude']].append(test_locations[['customer_id', 'latitude']]
                                ).groupby('customer_id').max() - train_locations[['customer_id', 'latitude']].append(
                                test_locations[['customer_id', 'latitude']]).groupby('customer_id').min()).latitude.map(abs).to_dict()
cust_location_lon_range = (train_locations[['customer_id', 'longitude']].append(test_locations[['customer_id', 'longitude']]
                                ).groupby('customer_id').max() - train_locations[['customer_id', 'longitude']].append(
                                test_locations[['customer_id', 'longitude']]).groupby('customer_id').min()).longitude.map(abs).to_dict()
train['cust_location_lat_range'] = train.customer_id.map(cust_location_lat_range)
test['cust_location_lat_range'] = test.customer_id.map(cust_location_lat_range)
train['cust_location_lon_range'] = train.customer_id.map(cust_location_lon_range)
test['cust_location_lon_range'] = test.customer_id.map(cust_location_lon_range)

In [76]:
vendor_location_lat_range = (orders.merge(train_locations[['customer_id', 'latitude']], on = 'customer_id')[['vendor_id', 'latitude']]
                            .groupby('vendor_id').max() - orders.merge(train_locations[['customer_id', 'latitude']], 
                            on = 'customer_id')[['vendor_id', 'latitude']].groupby('vendor_id').min()).latitude.map(abs).to_dict()
vendor_location_lon_range = (orders.merge(train_locations[['customer_id', 'longitude']], on = 'customer_id')[['vendor_id', 'longitude']].groupby('vendor_id').max() - orders.merge(train_locations[['customer_id', 'longitude']], 
                            on = 'customer_id')[['vendor_id', 'longitude']].groupby('vendor_id').min()).longitude.map(abs).to_dict()
train['vendor_location_lat_range'] = train.vendor_id.map(vendor_location_lat_range)
test['vendor_location_lat_range'] = test.vendor_id.map(vendor_location_lat_range)
train['vendor_location_lon_range'] = train.vendor_id.map(vendor_location_lon_range)
test['vendor_location_lon_range'] = test.vendor_id.map(vendor_location_lon_range)

In [77]:
train['member_since_diff'] = train['member_since'] - train['vendor_member_since']
test['member_since_diff'] = test['member_since'] - test['vendor_member_since']

In [78]:
df = orders[['vendor_id', 'customer_id', 'akeed_order_id']].sort_values(by = ['vendor_id', 'customer_id', 'akeed_order_id'])
df['prev_order_id'] = df.groupby(['vendor_id', 'customer_id']).shift(1)
df['prev_order_id_diff'] =  df['akeed_order_id'] - df['prev_order_id']
vendor_order_frequency = df[['vendor_id', 'prev_order_id_diff']].groupby('vendor_id').mean().prev_order_id_diff.to_dict()
train['vendor_order_frequency'] = train['vendor_id'].map(vendor_order_frequency)
test['vendor_order_frequency'] = test['vendor_id'].map(vendor_order_frequency)

repeat_customers = df[df.prev_order_id.isnull()==False].drop_duplicates()['vendor_id'].value_counts().to_dict()
train['repeat_customers'] = train['vendor_id'].map(repeat_customers)
test['repeat_customers'] = test['vendor_id'].map(repeat_customers)


In [79]:
def get_latent_features(df, u,v, key, dim = 16):
    """
    u = feature whose latent vector is to be calculate
    u = feature against whole the latent vector is calculated
    t = time var
    """
    df = df.sort_values(by = [u,v])
    df['c'] = 1    
    R_df = df[[u,v,'c']].groupby([u,v]).sum().reset_index()
    R_df.columns = [u, v, 'tf']
    n = df[u].nunique()
    R_df['idf'] = R_df[[u, v]].groupby(v).transform('count')[u].map(lambda x : np.log(n/x))
    R_df['tf_idf'] = R_df['tf']*R_df['idf']
#     R_df['tf_idf'] = R_df['tf_idf'] - R_df[[u, 'tf_idf']].groupby(u).transform('mean').tf_idf
    
    u_ids = sorted(df[u].unique())
    v_ids = sorted(df[v].unique())
    u_dict = {el : i for i, el in enumerate(u_ids)}
    v_dict = {el : i for i, el in enumerate(v_ids)}

    R_df['u_id'] = R_df[u].map(u_dict)
    R_df['v_id'] = R_df[v].map(v_dict)
    rows = R_df.u_id.values
    cols = R_df.v_id.values
    data = R_df.tf_idf.values
    R = coo_matrix((data, (rows, cols)), shape=(len(u_ids), len(v_ids)))
#     U, sigma, Vt = svds(R, k = dim)
    model = NMF(n_components=dim, init = "random", random_state=0)
    U = model.fit_transform(R)
    Vt = model.components_
    vendor_latent_features = pd.DataFrame(Vt).T
    vendor_latent_features.columns = ['{}_latent_feature_{}'.format(key,i) for i in range(vendor_latent_features.shape[1])]
    vendor_latent_features[v] = vendor_latent_features.index.map(lambda x : v_ids[x])
    return vendor_latent_features

vendor_latent_features = get_latent_features(orders, 'customer_id', 'vendor_id', 'vendor', 8)


In [80]:
train = train.merge(vendor_latent_features, on = 'vendor_id')
test = test.merge(vendor_latent_features, on = 'vendor_id')

In [81]:
df = train_locations[['customer_id', 'latitude', 'longitude']].append(test_locations[['customer_id', 'latitude', 'longitude']])
df.latitude = df.latitude.map(lambda x : np.round(x,2))
df.longitude = df.longitude.map(lambda x : np.round(x,2))
df['num_cust_on_location'] = df[['latitude', 'longitude', 'customer_id']].groupby(['latitude', 'longitude']).transform('count')
num_cust_on_location = df.set_index('customer_id').num_cust_on_location.to_dict()
train['num_cust_on_location'] = train.customer_id.map(num_cust_on_location)
test['num_cust_on_location'] = test.customer_id.map(num_cust_on_location)

In [145]:
customer_longitudes = train_locations[['customer_id', 'location_number', 'longitude']].append(test_locations[['customer_id', 
                                    'location_number', 'longitude']]).set_index(['customer_id', 'location_number']).longitude.to_dict()
customer_latitudes = train_locations[['customer_id', 'location_number', 'latitude']].append(test_locations[['customer_id', 
                                    'location_number', 'latitude']]).set_index(['customer_id', 'location_number']).latitude.to_dict()

vendor_longitudes = vendors[['id', 'longitude']].set_index('id').longitude.to_dict()
vendor_latitudes = vendors[['id', 'latitude']].set_index('id').latitude.to_dict()

location_graph = nx.Graph()
for customer, location_number in tqdm(customer_longitudes.keys()):
    for vendor in vendor_longitudes.keys():
        lat1 = customer_latitudes[(customer, location_number)]
        lon1 = customer_longitudes[(customer, location_number)]
        lat2 = vendor_latitudes[vendor]
        lon2 = vendor_longitudes[vendor]
        
        if np.sqrt(((lat1 - lat2)**2 + (lon1 - lon2)**2)/2) <  0.025:
            location_graph.add_edge((customer, location_number), vendor)
            

In [147]:
len(list(nx.connected_components(location_graph)))

31

In [158]:
cc = {n : cc for cc, n_set in enumerate(list(nx.connected_components(location_graph))) for n in n_set}
train['cust_cc'] = [cc.get((row.customer_id, row.location_number), np.nan) for row in tqdm(train.itertuples())]
test['cust_cc'] = [cc.get((row.customer_id, row.location_number),np.nan) for row in tqdm(test.itertuples())]

train['vendor_cc'] = train.vendor_id.map(cc)
test['vendor_cc'] = test.vendor_id.map(cc)


In [159]:
train['cust_vendor_in_same_cc'] = 1*(train.cust_cc == train.vendor_cc)
test['cust_vendor_in_same_cc'] = 1*(test.cust_cc == test.vendor_cc)


In [164]:
print(list(train.columns))

['customer_id', 'location_number', 'location_type', 'latitude', 'longitude', 'vendor_id', 'authentication_id', 'vendor_latitude', 'vendor_longitude', 'vendor_category_en', 'vendor_category_id', 'delivery_charge', 'serving_distance', 'is_open', 'OpeningTime', 'OpeningTime2', 'prepration_time', 'commission', 'is_akeed_delivering', 'discount_percentage', 'status', 'verified', 'rank', 'language', 'vendor_rating', 'sunday_from_time1', 'sunday_to_time1', 'sunday_from_time2', 'sunday_to_time2', 'monday_from_time1', 'monday_to_time1', 'monday_from_time2', 'monday_to_time2', 'tuesday_from_time1', 'tuesday_to_time1', 'tuesday_from_time2', 'tuesday_to_time2', 'wednesday_from_time1', 'wednesday_to_time1', 'wednesday_from_time2', 'wednesday_to_time2', 'thursday_from_time1', 'thursday_to_time1', 'thursday_from_time2', 'thursday_to_time2', 'friday_from_time1', 'friday_to_time1', 'friday_from_time2', 'friday_to_time2', 'saturday_from_time1', 'saturday_to_time1', 'saturday_from_time2', 'saturday_to_tim

In [165]:
indep_vars = ['location_number', 'latitude', 'longitude', 'vendor_id', 'vendor_latitude', 'vendor_longitude', 
            'delivery_charge', 'serving_distance', 'is_open', 'prepration_time', 'discount_percentage', 'status', 'verified', 
              'rank', 'vendor_rating', 'sunday_from_time1', 'sunday_to_time1', 'sunday_from_time2', 'sunday_to_time2', 
              'monday_from_time1', 'monday_to_time1', 'monday_from_time2', 'monday_to_time2', 'tuesday_from_time1', 
              'tuesday_to_time1', 'tuesday_from_time2', 'tuesday_to_time2', 'wednesday_from_time1', 'wednesday_to_time1',
              'wednesday_from_time2', 'wednesday_to_time2', 'thursday_from_time1', 'thursday_to_time1', 'thursday_from_time2', 
              'thursday_to_time2', 'friday_from_time1', 'friday_to_time1', 'friday_from_time2', 'friday_to_time2',
              'saturday_from_time1', 'saturday_to_time1', 'saturday_from_time2', 'saturday_to_time2', 'primary_tags', 
              'device_type', 'vendor_tags_svd_0', 'vendor_tags_svd_1', 'vendor_tags_svd_2', 'vendor_tags_svd_3',
              'vendor_tags_svd_4', 'vendor_tags_svd_5', 'vendor_tags_svd_6', 'vendor_tags_svd_7', 'vendor_member_since', 
              'customer_is_female', 'customer_age', 'customer_num_updates', 'member_since', 'location_cluster', 
              'vendor_location_cluster_order_count', 'longitude_corrected', 'latitude_corrected', 'vendor_longitude_corrected',
              'vendor_latitude_corrected', 'vendor_distance', 'vendor_distance_corrected', 'lon_diff', 'lat_diff', 
              'lon_diff_corrected', 'lat_diff_corrected', 'lon_diff_abs', 'lat_diff_abs', 'lon_diff_corrected_abs', 
              'lat_diff_corrected_abs', 'customer_location_counts', 'num_vendors_nearby', 'distance_rank', 'distance_rank_lat',
              'distance_rank_lon', 'vendor_uniuqe_customer_count', 'vendor_entropy', 'vendor_total_amount_by_location_type', 
              'daily_order_counts_mean', 'daily_order_counts_std', 'vendor_ratings_mean', 'vendor_ratings_std',
              'vendor_ratings_count', 'vendor_driver_ratings_mean', 'vendor_driver_ratings_std', 'vendor_driver_ratings_count',
              'vendor_is_restaurant', 'location_type_is_home', 'location_type_is_work', 'location_type_is_other', 
              'vendor_mean_delivery_distance', 'vendor_std_delivery_distance', 'vendor_max_delivery_distance', 
              'cust_location_lat_range', 'cust_location_lon_range', 'vendor_location_lat_range', 'vendor_location_lon_range',
              'member_since_diff', 'vendor_order_frequency', 'repeat_customers', 'vendor_latent_feature_0',
              'vendor_latent_feature_1', 'vendor_latent_feature_2', 'vendor_latent_feature_3', 'vendor_latent_feature_4', 
              'vendor_latent_feature_5', 'vendor_latent_feature_6', 'vendor_latent_feature_7', 'num_cust_on_location',
              'vendor_updated_days_after_created', 'vendor_updated_since', 'updated_since', 'updated_created_diff', 
              'cust_cc', 'vendor_cc', 'cust_vendor_in_same_cc'
             ]
len(indep_vars)

120

In [167]:
def lgb_f1_score(y_hat, data):
    return 'f1', f1_score(data.get_label(), y_hat > 0.125), True
params = {
    'task': 'train',
    'boosting_type': 'gbdt',
    'objective':'binary',
    'metric': {'auc'},
    'num_leaves': 256,
    'learning_rate': 0.01,
    'feature_fraction': 0.65,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'verbose': 1,
    'min_data_in_leaf' : 5,
    'max_bin' : 255,
    'lambda_l1' : 0.00002,
    'lambda_l2' : 0.00001,
    'min_gain_to_split' : 0.001
}
import random
random.seed(2)
models = []
for fold in sorted(validation_k_folds.fold.unique()):
    print("fold =", fold)
    dev_customers = set(train.customer_id) - set(validation_k_folds[validation_k_folds.fold == fold].customer_id)
    dev = train[train.customer_id.isin(dev_customers)]
    val = train[train.customer_id.isin(dev_customers) == False]

    print(dev.shape, val.shape)
    lgb_dev = lgb.Dataset(dev[indep_vars], dev['target'])
    lgb_val = lgb.Dataset(val[indep_vars], val['target'])

    model = lgb.train(params, lgb_dev, num_boost_round = 3000, valid_sets = (lgb_val), early_stopping_rounds = 500,
                         feval=lgb_f1_score, verbose_eval = 100)
    models.append(model)

fold = 1
(5107400, 140) (842900, 140)
Training until validation scores don't improve for 500 rounds
[100]	valid_0's auc: 0.917432	valid_0's f1: 0.208236
[200]	valid_0's auc: 0.920577	valid_0's f1: 0.242664
[300]	valid_0's auc: 0.922142	valid_0's f1: 0.249525
[400]	valid_0's auc: 0.92299	valid_0's f1: 0.250563
[500]	valid_0's auc: 0.923434	valid_0's f1: 0.2508
[600]	valid_0's auc: 0.923856	valid_0's f1: 0.251084
[700]	valid_0's auc: 0.924041	valid_0's f1: 0.252622
[800]	valid_0's auc: 0.92427	valid_0's f1: 0.253072
[900]	valid_0's auc: 0.924464	valid_0's f1: 0.253032
[1000]	valid_0's auc: 0.924604	valid_0's f1: 0.253361
[1100]	valid_0's auc: 0.9247	valid_0's f1: 0.253475
[1200]	valid_0's auc: 0.924835	valid_0's f1: 0.254056
[1300]	valid_0's auc: 0.924916	valid_0's f1: 0.253728
[1400]	valid_0's auc: 0.925034	valid_0's f1: 0.254278
[1500]	valid_0's auc: 0.925135	valid_0's f1: 0.254249
[1600]	valid_0's auc: 0.925231	valid_0's f1: 0.254237
[1700]	valid_0's auc: 0.925294	valid_0's f1: 0.2541

Early stopping, best iteration is:
[1659]	valid_0's auc: 0.924014	valid_0's f1: 0.251113
fold = 7
(5090700, 140) (859600, 140)
Training until validation scores don't improve for 500 rounds
[100]	valid_0's auc: 0.917514	valid_0's f1: 0.215441
[200]	valid_0's auc: 0.920934	valid_0's f1: 0.251824
[300]	valid_0's auc: 0.922569	valid_0's f1: 0.256726
[400]	valid_0's auc: 0.923476	valid_0's f1: 0.258587
[500]	valid_0's auc: 0.924152	valid_0's f1: 0.259045
[600]	valid_0's auc: 0.924577	valid_0's f1: 0.259534
[700]	valid_0's auc: 0.924803	valid_0's f1: 0.259832
[800]	valid_0's auc: 0.924981	valid_0's f1: 0.260235
[900]	valid_0's auc: 0.925123	valid_0's f1: 0.259949
[1000]	valid_0's auc: 0.925264	valid_0's f1: 0.26002
[1100]	valid_0's auc: 0.925383	valid_0's f1: 0.260385
[1200]	valid_0's auc: 0.925472	valid_0's f1: 0.260418
[1300]	valid_0's auc: 0.925576	valid_0's f1: 0.260295
[1400]	valid_0's auc: 0.925644	valid_0's f1: 0.260187
[1500]	valid_0's auc: 0.925735	valid_0's f1: 0.259663
[1600]	vali

In [84]:
def lgb_f1_score(y_hat, data):
    return 'f1', f1_score(data.get_label(), y_hat > 0.125), True
params = {
    'task': 'train',
    'boosting_type': 'gbdt',
    'objective':'binary',
    'metric': {'auc'},
    'num_leaves': 256,
    'learning_rate': 0.01,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'verbose': 1,
    'min_data_in_leaf' : 2,
    'max_bin' : 255,
    'lambda_l1' : 0.00002,
    'lambda_l2' : 0.00001,
    'min_gain_to_split' : 0.001
}
import random
random.seed(2)
models = []
for fold in sorted(validation_k_folds.fold.unique()):
    print("fold =", fold)
    dev_customers = set(train.customer_id) - set(validation_k_folds[validation_k_folds.fold == fold].customer_id)
    dev = train[train.customer_id.isin(dev_customers)]
    val = train[train.customer_id.isin(dev_customers) == False]

    print(dev.shape, val.shape)
    lgb_dev = lgb.Dataset(dev[indep_vars], dev['target'])
    lgb_val = lgb.Dataset(val[indep_vars], val['target'])

    model = lgb.train(params, lgb_dev, num_boost_round = 3000, valid_sets = (lgb_val), early_stopping_rounds = 500,
                         feval=lgb_f1_score, verbose_eval = 100)
    models.append(model)

fold = 1
(5107400, 137) (842900, 137)
Training until validation scores don't improve for 500 rounds
[100]	valid_0's auc: 0.917482	valid_0's f1: 0.20946
[200]	valid_0's auc: 0.920472	valid_0's f1: 0.242755
[300]	valid_0's auc: 0.921996	valid_0's f1: 0.249374
[400]	valid_0's auc: 0.922808	valid_0's f1: 0.251118
[500]	valid_0's auc: 0.923341	valid_0's f1: 0.25122
[600]	valid_0's auc: 0.923678	valid_0's f1: 0.251178
[700]	valid_0's auc: 0.923938	valid_0's f1: 0.251957
[800]	valid_0's auc: 0.924155	valid_0's f1: 0.252488
[900]	valid_0's auc: 0.924405	valid_0's f1: 0.252336
[1000]	valid_0's auc: 0.924535	valid_0's f1: 0.252152
[1100]	valid_0's auc: 0.924641	valid_0's f1: 0.252772
[1200]	valid_0's auc: 0.924757	valid_0's f1: 0.253269
[1300]	valid_0's auc: 0.92481	valid_0's f1: 0.252869
[1400]	valid_0's auc: 0.924932	valid_0's f1: 0.253314
[1500]	valid_0's auc: 0.925003	valid_0's f1: 0.253091
[1600]	valid_0's auc: 0.92508	valid_0's f1: 0.252429
[1700]	valid_0's auc: 0.925141	valid_0's f1: 0.25

[200]	valid_0's auc: 0.920475	valid_0's f1: 0.250142
[300]	valid_0's auc: 0.922083	valid_0's f1: 0.256096
[400]	valid_0's auc: 0.923002	valid_0's f1: 0.256389
[500]	valid_0's auc: 0.92359	valid_0's f1: 0.257303
[600]	valid_0's auc: 0.924	valid_0's f1: 0.257362
[700]	valid_0's auc: 0.924249	valid_0's f1: 0.257865
[800]	valid_0's auc: 0.9245	valid_0's f1: 0.258244
[900]	valid_0's auc: 0.924672	valid_0's f1: 0.258899
[1000]	valid_0's auc: 0.924805	valid_0's f1: 0.259193
[1100]	valid_0's auc: 0.924927	valid_0's f1: 0.259379
[1200]	valid_0's auc: 0.925027	valid_0's f1: 0.259836
[1300]	valid_0's auc: 0.925099	valid_0's f1: 0.259668
[1400]	valid_0's auc: 0.925172	valid_0's f1: 0.25908
[1500]	valid_0's auc: 0.925251	valid_0's f1: 0.258959
[1600]	valid_0's auc: 0.925314	valid_0's f1: 0.258774
[1700]	valid_0's auc: 0.925359	valid_0's f1: 0.258873
Early stopping, best iteration is:
[1279]	valid_0's auc: 0.925089	valid_0's f1: 0.260185


In [168]:
val1 = []
for i, fold in tqdm(enumerate(sorted(validation_k_folds.fold.unique()))):
    dev_customers = set(train.customer_id) - set(validation_k_folds[validation_k_folds.fold == fold].customer_id)
    val = train[train.customer_id.isin(dev_customers) == False]
    val['pred'] = models[i].predict(val[indep_vars])
#     val['pred'] = train.target.mean() * val['pred'] / val['pred'].mean()
    val1.append(val)
val1 = pd.concat(val1)   

/home/ubuntu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """


In [94]:
for i in range(1,11):
    th = 0.12 + i/1000
    print(th, f1_score(val1.target, val1.pred.map(lambda x : 1 if x > th else 0)))

0.121 0.2543285304887757
0.122 0.25443890887380405
0.123 0.2547014253487502
0.124 0.2548197991537952
0.125 0.25507177845053125
0.126 0.25456467771276353
0.127 0.254373912725487
0.128 0.2541918705350327
0.129 0.2541063160630281
0.13 0.25394636261098724


In [169]:
preds = []
for i in tqdm(range(5)):
    pred = models[i].predict(test[indep_vars])
    preds.append(pred)

In [170]:
test['pred'] = list(np.array(preds).mean(axis = 0))

In [171]:
test['target'] = test.pred.map(lambda x : 1 if x > 0.125 else 0)

In [172]:
test['CID X LOC_NUM X VENDOR'] = test.customer_id + " X " + test.location_number.astype(str) + " X " + test.vendor_id.astype(str)
test[['CID X LOC_NUM X VENDOR', 'target']].to_csv("./submission_{}.csv".format(version), index = False)
test[['CID X LOC_NUM X VENDOR', 'pred']].to_csv("./preds_{}.csv".format(version), index = False)

In [173]:
val1[['customer_id', 'location_number', 'vendor_id', 'target', 'pred']].to_csv("./val_preds_{}.csv".format(version), index = False)